# EXIST 2026 — Task 3 — Calibração pós-treino + diagnóstico dos NaN (Combo C)

**Zero GPU de treino.** Roda sobre um checkpoint já treinado: um `predict` na validação,
depois aritmética. Minutos, não horas.

**Item 4 do doc 17 / opção 6 do doc 20** — a calibração estava explicitamente barrada até
o probing de heads terminar. O probing terminou (doc 21) e as duas rotas de realocação de
treino entre tasks fecharam (docs 18 §9.5 e 22 §6). A calibração é **ortogonal** ao
gargalo identificado: não muda a representação, muda como as probabilidades são
espalhadas — que é o que o ICM-Soft mede.

**Item 9 do doc 17** — a última seção diagnostica por que `fmeasure_hard` (B e C) e
`crossentropy_soft` (B e C) vêm `nan` em toda run. Diagnóstico puro, não corrige nada.

## Protocolo anti-leakage

Temperatura ajustada numa metade da validação e **reportada na outra**, com o mesmo
`vc.split_val_halves(n, SPLIT_SEED)` de `tune_task_c_thresholds` (ARCH-11 b). O número
reportável é sempre `after_on_report_half`. O da metade de ajuste é impresso só para
diagnosticar overfit da própria calibração.

⚠ **Escalar em A/B, vector scaling na C — PRÉ-REGISTRADO** no item 4 do doc 17, escrito
antes de qualquer medição. A variante escalar da C é gravada só como registro; escolher
entre as duas olhando a métrica de reporte seria leakage.

⚠ **Assimetria a declarar na tese:** estes números vêm de ~metade do val; os de
`compute_metrics` vêm do val inteiro. Não colocar lado a lado sem a nota.

## 1. Imports e escolha da run

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import TrainingArguments, XLMRobertaTokenizer

import video_core as vc
import calibration as cal

In [ ]:
# Qualquer run treinada serve — o notebook detecta o encoder pelo meta.json.
# Candidatas: run_B2_bio_viewer_seed42_CLIP (braço CLIP oficial)
#             run_B2_bio_viewer_seed42_B15  (baseline VideoMAE)
RUN_DIR = Path("run_B2_bio_viewer_seed42_CLIP")

assert RUN_DIR.exists(), f"run inexistente: {RUN_DIR}"
meta = json.load(open(RUN_DIR / "meta.json"))
ENCODER = meta.get("video_encoder", "videomae")
AUDIO_VERSION = meta.get("audio_version", "v2")
INIT_SEED = meta.get("init_seed", vc.INIT_SEED)

print(f"run = {RUN_DIR}")
print(f"encoder = {ENCODER} | áudio = {AUDIO_VERSION} | init_seed = {INIT_SEED}")
print(f"use_bio = {meta['use_bio']} | use_viewer_emb = {meta['use_viewer_emb']} | "
      f"fusion_in = {meta['fusion_in']}")

## 2. Dados — split congelado, scaler do checkpoint

O `RobustScaler` vem do checkpoint, **nunca** re-ajustado (BUG-B/ARCH-18).

In [ ]:
assert vc.split_artifact_path().exists(), (
    "Copiar Dataset/val_split_42.json para a VM antes de rodar.")

full_df = pd.read_json("./Dataset/training/EXIST2026_training.json").transpose()
train_df, val_df = vc.build_frozen_split(full_df)
print(f"Treino: {len(train_df)} | Val: {len(val_df)}")

if ENCODER == "clip":
    import video_core_clip as vcc
    model, scaler = vcc.load_run_clip(RUN_DIR, init_seed=INIT_SEED)
    DatasetCls = vcc.EXISTVideoDatasetCLIP
else:
    model, scaler = vc.load_run(RUN_DIR, init_seed=INIT_SEED)
    DatasetCls = vc.EXISTVideoDataset

assert scaler is not None, "scaler.pkl ausente — não ajustar um novo (leakage)"

tokenizer = XLMRobertaTokenizer.from_pretrained(vc.XLMR_NAME)
val_dataset = DatasetCls(
    val_df, tokenizer, scaler,
    vc.frames_dir("train"), vc.audio_dir("train", version=AUDIO_VERSION),
    max_length=vc.MAX_LENGTH)
print(f"Val: {len(val_dataset)} amostras | dataset = {DatasetCls.__name__}")

## 3. Trainer só para `predict`

Nenhum treino acontece aqui. `TrainingArguments` mínima, só para o `predict` rodar com o
mesmo collator e as mesmas `label_names` do treino.

In [ ]:
args = TrainingArguments(
    output_dir="./_calib_tmp",
    per_device_eval_batch_size=2,
    bf16=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    label_names=["labels_a", "labels_b", "labels_c"],
    report_to="none",
)

trainer = vc.MultiTaskTrainer(
    model=model,
    args=args,
    eval_dataset=val_dataset,
    compute_metrics=vc.compute_metrics,
    data_collator=vc.video_collate,
)
trainer.label_names = ["labels_a", "labels_b", "labels_c"]
print("trainer pronto (só para predict — nenhum treino)")

## 4. Calibração (item 4)

Uma passagem de `predict`, temperatura ajustada por NLL soft na metade de ajuste,
`ICMSoftNorm` medido na metade de reporte antes e depois.

In [ ]:
calib = cal.calibrate_run(
    trainer, val_dataset,
    save_path=str(RUN_DIR / "calibration.json"),
)

### 4.1 Como ler o resultado

- **Só o `Δ` da linha REPORTE é reportável.** Positivo = a calibração ajuda.
- **Compare os dois deltas.** Ganho no ajuste que não sobrevive no reporte = a
  temperatura ajustou ruído; reportar como nulo. A queda da NLL no ajuste **não** é
  evidência de nada — é o objetivo que foi otimizado.
- **`T ≈ 1` significa que o modelo já estava calibrado** naquela task: resultado
  legítimo e informativo, não falha do método.
- **`T > 1`** = modelo over-confident (probabilidades espremidas nos extremos);
  **`T < 1`** = under-confident.
- **Teste de resposta conhecida, de graça:** métricas *hard* são invariantes a
  temperature scaling por construção (argmax em B; o cruzamento de 0,5 da sigmoide em
  A/C fica em logit 0 para qualquer `T>0`). Se alguma métrica hard se mexer, o problema
  é no pipeline, não na calibração.
- n=1 seed, ~252 amostras por metade. Deltas pequenos são ruído.

## 5. Diagnóstico dos NaN do PyEvALL (item 9)

`vc._extract_metrics_from_report` devolve `nan` em dois casos que não distingue: a métrica
não veio no relatório, ou veio com `status != 'OK'` (ARCH-14 — devolver `nan` em vez de
omitir foi deliberado, para a coluna não sumir da tabela do Trainer). Esta célula abre o
relatório cru e diz qual dos dois é, por métrica.

⏱ Esta célula roda um segundo `predict` completo sobre a validação (as duas seções são
independentes de propósito — o diagnóstico não deve depender da calibração ter rodado).
Conte alguns minutos, como a célula 4.

In [ ]:
nan_findings = cal.diagnose_pyevall_nans(trainer, val_dataset)

with open(RUN_DIR / "pyevall_nan_diagnosis.json", "w") as f:
    json.dump(nan_findings, f, indent=2, default=str)
print(f"\n✓ gravado em {RUN_DIR / 'pyevall_nan_diagnosis.json'}")

## 6. Próximo passo

Rodar este notebook também sobre `run_B2_bio_viewer_seed42_B15` (basta trocar `RUN_DIR`
na célula 1 — o resto se adapta pelo `meta.json`). Se a calibração ajudar nos dois
encoders, o efeito é do método; se ajudar só num, é do checkpoint.

⚠ Não ajustar nada olhando o resultado da metade de reporte. A temperatura sai da metade
de ajuste, ponto.